## J. Production Reliability

### 147. What happens if the **LLM API goes down**?

> **Answer:** “I detect failures through timeout/health monitoring, apply bounded retries for transient errors, and then fail over to a validated fallback model. If the operation is non-critical, I can queue it for later processing; for critical actions, I fail safely or trigger HIL.”

```text
LLM Request
    ↓
Primary LLM
    ↓
Failure?
 ├── No → Response
 └── Yes
      ↓
   Retry
      ↓
   Fails?
      ↓
 Fallback LLM
      ↓
 Still fails?
      ↓
 Queue / HIL / Graceful Failure
```

**One-liner:**  
> **“I use retry → fallback → queue/HIL, with graceful degradation rather than allowing the failure to propagate to the business transaction.”**

---

### 148. What happens if the **vector database goes down**?

> **Answer:** “I treat the vector database as a dependency failure. I use health checks, timeout controls and a fallback retrieval strategy. Depending on the business requirement, I can use a replicated search service, cached results, keyword search, or temporarily return a controlled ‘knowledge unavailable’ response.”

```text
Query
 ↓
Vector DB
 ↓
Available?
 ├── Yes → Retrieve
 └── No
      ↓
   Fallback Search / Cache
      ↓
   Available?
    ├── Yes → Continue
    └── No  → Graceful Failure
```

**One-liner:**  
> **“I don't allow vector-store failure to cause uncontrolled agent behavior; I use fallback retrieval or graceful degradation depending on the SLA.”**

---

### 149. What happens if a **downstream business API fails**?

> **Answer:** “I distinguish between transient and permanent failures. For transient failures I retry with exponential backoff; if retries fail, I use a circuit breaker and move the operation to a retry queue/DLQ. I also make the business operation idempotent so retrying doesn't create duplicate transactions.”

```text
Agent
 ↓
Business API
 ↓
Failure
 ↓
Retry
 ↓
Still Failed
 ↓
Circuit Breaker
 ↓
Queue / DLQ
 ↓
Resume Later
```

**One-liner:**  
> **“The agent should never blindly retry a business transaction; I combine bounded retries, circuit breaking, idempotency and durable queues.”**

---

### 150. How do you implement **circuit breakers**?

> **Answer:** “A circuit breaker monitors repeated downstream failures and temporarily stops sending requests when the failure threshold is exceeded.”

```text
CLOSED
  ↓
Failures > Threshold
  ↓
OPEN
  ↓
Wait / Cooldown
  ↓
HALF-OPEN
  ↓
Test Request
 ┌──────┴──────┐
Success      Failure
 ↓              ↓
CLOSED         OPEN
```

### States

- **Closed** → requests flow normally
- **Open** → requests blocked immediately
- **Half-open** → allow limited test requests

**One-liner:**  
> **“Circuit breakers prevent cascading failures by stopping calls to an unhealthy dependency and allowing recovery before traffic is restored.”**

---

### 151. **Retry vs Fallback vs Circuit Breaker?**

| Mechanism | Purpose | When |
|---|---|---|
| **Retry** | Try the same operation again | Transient failure |
| **Fallback** | Use an alternative implementation/service | Primary unavailable |
| **Circuit Breaker** | Stop calling unhealthy service | Repeated failures |

```text
Request
 ↓
Retry
 ↓
Still failing?
 ↓
Fallback
 ↓
Repeated dependency failures
 ↓
Circuit Breaker
```

**One-liner:**  
> **“Retry handles transient failures, fallback provides an alternative path, and circuit breaker protects the system from repeatedly calling an unhealthy dependency.”**

---

### 152. How do you make tool execution **idempotent**?

> **Answer:** “I assign every business operation a unique **idempotency key** and persist the operation status before execution. If the same request arrives again, I return the existing result instead of executing the business action again.”

```text
Request
 ↓
Idempotency Key
 ↓
Already Processed?
 ├── Yes → Return Existing Result
 └── No
      ↓
   Execute Tool
      ↓
   Store Result
```

Example:

```python
idempotency_key = f"{tenant_id}:{request_id}"
```

For an API:

```text
POST /shift
Idempotency-Key: REQ-12345
```

**One-liner:**  
> **“Idempotency means the same business request can safely be executed multiple times without creating multiple side effects.”**

---

### 153. How do you prevent **duplicate business transactions**?

> **Answer:** “I use idempotency keys combined with a persistent transaction record and database uniqueness constraints.”

```text
Request
 ↓
Generate Transaction ID
 ↓
Check Transaction Store
 ↓
Exists?
 ├── Yes → Return Existing Status
 └── No
      ↓
   Execute
      ↓
   Commit Transaction
      ↓
   Store SUCCESS
```

### Controls

- Idempotency key
- Unique transaction ID
- Database unique constraint
- Atomic transaction
- Status tracking
- Distributed lock where necessary

**One-liner:**  
> **“I enforce uniqueness at both the application and database layers, so even concurrent requests cannot create duplicate business transactions.”**

---

### 154. How do you handle **partial workflow failure**?

> **Answer:** “I persist workflow state after important steps, make each step idempotent, and distinguish completed from failed steps. The workflow can resume from the last successful checkpoint rather than restarting everything.”

```text
Step 1 ✓
  ↓
Step 2 ✓
  ↓
Step 3 ✗
  ↓
Checkpoint
  ↓
Retry / Resume
  ↓
Step 3
  ↓
Step 4
```

### Controls

- Checkpointing
- Durable state
- Idempotent tools
- Retry policies
- Compensation/rollback where required
- DLQ
- HIL for unrecoverable cases

**One-liner:**  
> **“I persist state at workflow boundaries and resume from the last successful checkpoint, while making already-completed business actions idempotent.”**

---

### 155. How do you implement **dead-letter queues**?

> **Answer:** “I use a DLQ for messages that repeatedly fail processing after the configured retry limit. I preserve the original message, error details and metadata so the issue can be investigated and replayed safely.”

```text
Queue
 ↓
Consumer
 ↓
Processing
 ↓
Failure
 ↓
Retry
 ↓
Retry Limit Exceeded
 ↓
DLQ
 ↓
Monitor / Investigate
 ↓
Fix
 ↓
Replay
```

### DLQ message should contain

```text
message_id
workflow_id
tenant_id
original_payload/reference
error
retry_count
timestamp
```

**One-liner:**  
> **“DLQ prevents poison messages from continuously blocking the main queue while preserving enough information for investigation and controlled replay.”**

---

### 156. How do you resume failed **LangGraph workflows**?

> **Answer:** “I use LangGraph checkpointing with a persistent checkpointer and a unique thread/workflow ID. The graph state is persisted at checkpoints, so after a failure I can resume from the last persisted state rather than restarting the entire workflow.”

```text
LangGraph
   ↓
Node A ✓
   ↓
Checkpoint
   ↓
Node B ✓
   ↓
Checkpoint
   ↓
Node C ✗
   ↓
Failure
```

After recovery:

```text
Checkpoint
   ↓
Resume Node C
   ↓
Node D
   ↓
END
```

### Production approach

- Persistent checkpointer
- Unique `thread_id`
- Durable state
- Retry policies
- Idempotent tools
- HIL for blocked workflows

**One-liner:**  
> **“I persist LangGraph state using a durable checkpointer and resume using the same thread ID, so failures don't require replaying completed business actions.”**

---

### 157. How do you design **disaster recovery**?

> **Answer:** “I start with business-defined **RTO and RPO**, then design redundancy, backups, replication and recovery procedures around those requirements.”

```text
Primary Region
      ↓
Replication / Backup
      ↓
Secondary Region
      ↓
Failover
      ↓
Recovery
```

### DR layers

**Application**
- Multi-zone deployment
- Stateless services
- Infrastructure as Code

**Data**
- Database backups
- Replication
- Blob versioning
- Search-index rebuild strategy
- Checkpoint persistence

**AI**
- Model fallback
- Alternative deployment/provider
- Prompt/model versioning
- Rebuildable embeddings/indexes

**Workflow**
- Durable queues
- DLQs
- LangGraph checkpoints
- Idempotency

**Operational**
- Health checks
- Automated failover
- Runbooks
- Periodic DR drills

### RTO vs RPO

```text
RTO → How quickly must service recover?
RPO → How much data loss is acceptable?
```

**One-liner:**  
> **“I design DR around RTO/RPO, with multi-zone or multi-region redundancy, replicated durable state, backups, rebuildable AI indexes, durable queues and tested failover procedures.”**

---

### 🔑 Production Reliability mental model

```text
Transient Failure
      ↓
    RETRY
      ↓
Persistent Failure
      ↓
   FALLBACK
      ↓
Repeated Dependency Failure
      ↓
CIRCUIT BREAKER
      ↓
Unprocessable Message
      ↓
    DLQ
      ↓
Workflow Failure
      ↓
CHECKPOINT + RESUME
      ↓
Business Transaction
      ↓
IDEMPOTENCY
      ↓
Disaster
      ↓
DR / FAILOVER
```
